# Chicago Crime Analysis
## Requête 4 — Analyse spatiale & Clustering
**Auteure : Chrisa**

**Objectif :** Identifier les zones géographiques de Chicago où la criminalité se concentre, grâce à la visualisation spatiale et aux algorithmes de clustering K-means et OPTICS.

---

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from sklearn.cluster import OPTICS, KMeans
import plotly.express as px
import warnings
warnings.filterwarnings("ignore")

### Chargement du dataset

On charge le CSV fourni par M1 (Angelikia).

**Point important :** les colonnes `Latitude` et `Longitude` utilisent des virgules comme séparateur décimal (format européen, ex: `41,838`). On remplace par des points avant conversion.

In [ ]:
# Fonction de chargement et nettoyage des données spatiales
# Input  : chemin vers le fichier CSV
# Output : dataframe avec Latitude/Longitude en float, NaN supprimés
def load_spatial_data(path="../data/chicago_crime.csv"):
    df = pd.read_csv(path)

    # Suppression des lignes sans coordonnées GPS
    df = df.dropna(subset=["Latitude", "Longitude"])

    # Correction du format décimal : virgule → point
    df["Latitude"]  = df["Latitude"].astype(str).str.replace(",", ".").astype(float)
    df["Longitude"] = df["Longitude"].astype(str).str.replace(",", ".").astype(float)

    # Filtre géographique : on garde uniquement les points dans Chicago
    df = df[
        df["Latitude"].between(41.6, 42.1) &
        df["Longitude"].between(-87.9, -87.5)
    ].copy()

    print(f"Dataset chargé : {len(df)} lignes avec coordonnées valides")
    return df

df = load_spatial_data()
df.head()

### Création des objets spatiaux (Geopandas)

On convertit chaque paire (latitude, longitude) en un objet `Point` géoréférencé.

In [ ]:
# Fonction de création du GeoDataFrame
# Input  : dataframe nettoyé avec colonnes Latitude / Longitude
# Output : GeoDataFrame avec colonne geometry (objets Point)
def create_geodataframe(df):
    geometry = [Point(lon, lat) for lon, lat in zip(df["Longitude"], df["Latitude"])]
    gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")
    print(f"GeoDataFrame créé : {len(gdf)} points | Projection : {gdf.crs}")
    return gdf

gdf = create_geodataframe(df)
gdf[["Primary Type", "Arrest", "Latitude", "Longitude", "geometry"]].head()

## Carte de densité des crimes

Ce graphique représente la concentration géographique des crimes à Chicago.
Les zones en rouge foncé correspondent aux endroits où le plus grand nombre d'incidents a été enregistré.

**Calcul :** Pour chaque point de la carte, Plotly compte le nombre de crimes dans un rayon proche et colore la zone en conséquence.

**Interprétation :** Plus la couleur est foncée, plus la densité de crimes est élevée dans cette zone.

In [ ]:
# Fonction de visualisation de la densité spatiale des crimes
# Input  : dataframe nettoyé avec colonnes Latitude et Longitude
# Output : carte de densité interactive Plotly
def plot_density_map(df):
    fig = px.density_mapbox(
        df,
        lat="Latitude",
        lon="Longitude",
        radius=10,
        center={"lat": 41.85, "lon": -87.65},
        zoom=10,
        mapbox_style="carto-positron",
        title="Carte de densité des crimes à Chicago",
        color_continuous_scale="Reds"
    )
    fig.update_layout(margin={"r": 0, "t": 40, "l": 0, "b": 0})
    return fig

fig_density = plot_density_map(df)
fig_density.show()

## Clustering K-means

K-means divise les crimes en `k` zones géographiques en regroupant les points les plus proches.
On choisit k=6 car Chicago est naturellement divisée en 6 grands secteurs (North, South, West, etc.).

**Calcul :** L'algorithme place 6 centres, assigne chaque crime au centre le plus proche, déplace les centres vers le milieu de leur groupe, et répète jusqu'à convergence.

**Interprétation :** Chaque couleur représente une zone géographique. Les zones avec le plus de points sont les quartiers les plus touchés par la criminalité.

In [ ]:
# Fonction d'application du clustering K-means sur les coordonnées GPS
# Input  : dataframe nettoyé, k = nombre de zones voulues
# Output : dataframe avec colonne cluster_kmeans ajoutée
def apply_kmeans(df, k=6):
    coords = df[["Latitude", "Longitude"]].values
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    df = df.copy()
    # .astype(str) pour que Plotly affiche des couleurs distinctes (et non un dégradé)
    df["cluster_kmeans"] = model.fit_predict(coords).astype(str)
    print(f"K-means : {k} zones créées")
    print(df["cluster_kmeans"].value_counts().sort_index())
    return df

# Fonction de visualisation des zones K-means sur la carte
# Input  : dataframe avec colonne cluster_kmeans
# Output : carte interactive colorée par zone
def plot_kmeans_map(df):
    fig = px.scatter_mapbox(
        df,
        lat="Latitude",
        lon="Longitude",
        color="cluster_kmeans",
        center={"lat": 41.85, "lon": -87.65},
        zoom=10,
        mapbox_style="carto-positron",
        title="K-means — 6 zones de criminalité à Chicago",
        hover_data=["Primary Type", "Arrest"],
        opacity=0.6
    )
    fig.update_layout(margin={"r": 0, "t": 40, "l": 0, "b": 0})
    return fig

df_kmeans = apply_kmeans(df, k=6)
fig_kmeans = plot_kmeans_map(df_kmeans)
fig_kmeans.show()

## Clustering OPTICS

Contrairement à K-means, OPTICS détecte automatiquement le nombre de zones denses
et étiquette les crimes isolés comme du bruit (label `-1`).

**Paramètres :**
- `min_samples=20` : minimum 20 crimes proches pour former une zone (adapté à 943 lignes)
- `xi=0.05` : sensibilité pour séparer les zones

**Interprétation :** Les zones colorées sont les hotspots denses. Les points `-1` sont des crimes isolés géographiquement.

In [ ]:
# Fonction d'application du clustering OPTICS sur les coordonnées GPS
# Input  : dataframe nettoyé, min_samples = densité minimale par zone
# Output : dataframe avec colonne cluster_optics ajoutée (-1 = bruit)
def apply_optics(df, min_samples=20):
    coords = df[["Latitude", "Longitude"]].values
    model = OPTICS(min_samples=min_samples, xi=0.05, min_cluster_size=0.05)
    df = df.copy()
    df["cluster_optics"] = model.fit_predict(coords).astype(str)
    nb_zones = df["cluster_optics"].nunique() - 1  # -1 pour exclure le bruit
    nb_bruit = (df["cluster_optics"] == "-1").sum()
    print(f"OPTICS : {nb_zones} zones denses | {nb_bruit} crimes isolés ({nb_bruit/len(df)*100:.1f}%)")
    print(df["cluster_optics"].value_counts())
    return df

# Fonction de visualisation des zones OPTICS sur la carte
# Input  : dataframe avec colonne cluster_optics
# Output : carte interactive colorée par zone (gris = bruit)
def plot_optics_map(df):
    fig = px.scatter_mapbox(
        df,
        lat="Latitude",
        lon="Longitude",
        color="cluster_optics",
        center={"lat": 41.85, "lon": -87.65},
        zoom=10,
        mapbox_style="carto-positron",
        title="OPTICS — Zones denses de criminalité à Chicago (−1 = crime isolé)",
        hover_data=["Primary Type", "Arrest"],
        opacity=0.6
    )
    fig.update_layout(margin={"r": 0, "t": 40, "l": 0, "b": 0})
    return fig

df_optics = apply_optics(df, min_samples=20)
fig_optics = plot_optics_map(df_optics)
fig_optics.show()

## Comparaison K-means vs OPTICS

| Critère | K-means | OPTICS |
|---|---|---|
| Nombre de zones | Fixé par l'utilisateur (k=6) | Découvert automatiquement |
| Crimes isolés | Forcés dans une zone | Étiquetés -1 (bruit) |
| Forme des zones | Sphérique | Libre |
| Vitesse | Rapide | Plus lent |

**Résultats observés :**  
Les deux méthodes s'accordent sur le fait que la criminalité est concentrée dans quelques zones précises du dataset.
K-means force tous les crimes dans une zone, ce qui donne une vision plus lissée.
OPTICS révèle les vrais hotspots denses et isole les crimes qui ne s'intègrent dans aucune zone.

**Limite observée :**  
Le dataset de M1 ne couvre qu'un seul quartier (Bridgeport, 943 lignes), ce qui limite la représentativité
de l'analyse spatiale. Voir `spatial_analysis_v2.ipynb` pour l'analyse sur toute la ville.

## Analyse complémentaire : types de crimes par zone

On enrichit le clustering K-means en regardant quels types de crimes dominent dans chaque zone.

**Interprétation :** Ce graphique permet de voir si certaines zones sont spécialisées dans un type de crime particulier.

In [ ]:
# Fonction de visualisation des types de crimes par zone K-means
# Input  : dataframe avec colonne cluster_kmeans
# Output : graphique barres empilées — composition criminelle par zone
def plot_crime_type_by_cluster(df):
    group = (
        df.groupby(["cluster_kmeans", "Primary Type"])
        .size()
        .reset_index(name="count")
    )
    fig = px.bar(
        group,
        x="cluster_kmeans",
        y="count",
        color="Primary Type",
        title="Types de crimes par zone géographique (K-means)",
        labels={"cluster_kmeans": "Zone", "count": "Nombre de crimes", "Primary Type": "Type de crime"},
        barmode="stack"
    )
    return fig

fig_types = plot_crime_type_by_cluster(df_kmeans)
fig_types.show()

## Synthèse

- La carte de densité confirme que la criminalité n'est pas uniformément répartie sur Chicago
- K-means identifie 6 zones géographiques avec des concentrations de crimes variables
- OPTICS détecte les vrais hotspots denses et révèle que certains crimes sont géographiquement isolés
- La composition des types de crimes varie selon les zones, ce qui suggère des profils criminels différents par quartier
- **Limite :** le dataset couvre un seul quartier — voir v2 pour l'analyse sur toute la ville

In [ ]:
# Main block — Requête 4 : Analyse spatiale (dataset M1)
# Auteure : Chrisa
# Input   : ../data/chicago_crime.csv (fourni par Angelikia)
# Output  : 4 visualisations interactives Plotly
if __name__ == "__main__":
    df          = load_spatial_data("../data/chicago_crime.csv")
    gdf         = create_geodataframe(df)
    plot_density_map(df).show()
    df_kmeans   = apply_kmeans(df, k=6)
    plot_kmeans_map(df_kmeans).show()
    df_optics   = apply_optics(df, min_samples=20)
    plot_optics_map(df_optics).show()
    plot_crime_type_by_cluster(df_kmeans).show()